# POPQUANTS | Strategy 2 — Triangulated Statistical Arbitrage
## เรียนแบบ Feynman: จากหุ้น 2 ตัวสู่เครือข่ายหุ้น 4 ตัว

**สำคัญ:** Notebook นี้เป็นการทดลอง **GICS/Economic-relationship baseline แบบ 4 หุ้น** ไม่ใช่ผลการทำซ้ำงานวิจัยเต็มของ Quantitativo (ยังไม่มี 13F/Asset Embeddings, pair-ranking ด้วย Sharpe, Depth≥4, Long/Short อย่างละ 25 ตัว)

อ่านข้อมูลจากไฟล์ uSMART CSV ที่ Strategy 1 สร้างไว้แล้วโดยไม่ Login ใหม่ ไม่ส่งคำสั่งซื้อขาย

**อ่านทีละ Cell:** ทุกครั้งเราจะ (1) เขียนสูตรด้วยคำง่าย ๆ (2) ใส่ตัวเลข (3) แสดงตาราง (4) ตรวจความหมาย

In [ ]:
# เตรียมไลบรารีและโฟลเดอร์ ไม่อ่านคีย์หรือส่งคำสั่งซื้อขาย
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from popquants_strategy2 import (read_prices, compute_signals, build_strategy, metrics, run)
DATA = Path.home() / 'Documents/usmart/popquants_strategy1/us_stock_adjusted_daily.csv'
OUT = Path.home() / 'Documents/usmart/popquants_strategy2'
UNIVERSE = ['AMAT','LRCX','KLAC','TER']
print('ไฟล์ราคา:',DATA)
print('Universe:',', '.join(UNIVERSE))

## STEP 1 — ราคาหุ้นมาจากไหน?

หุ้นในกลุ่ม Semiconductor Equipment มีบริบททางเศรษฐกิจร่วมกัน แต่ไม่ได้หมายความว่า Spread ต้องกลับสู่ค่าเฉลี่ยเสมอ

**ตารางที่ 1:** ราคาปิด *adjusted* ของหุ้น 4 ตัวในแต่ละวัน (ข้อมูลจริงขึ้นกับไฟล์บน Mac ของคุณ)

In [ ]:
prices = read_prices(DATA, UNIVERSE)
print('จำนวนวันที่ราคาตรงกัน:',len(prices),'|',prices.index.min().date(),'ถึง',prices.index.max().date())
display(prices.head(6).round(3))

## STEP 2 — ทำไมต้องใช้ Log-price Spread?

**สูตรเข้าใจง่าย:** `Spread ของ A กับ B = log(ราคาหุ้น A) − log(ราคาหุ้น B)`

\[S_{AB,t}=\ln(P_{A,t})-\ln(P_{B,t})\]

- `P_A` = ราคาปิด adjusted ของหุ้น A
- `P_B` = ราคาปิด adjusted ของหุ้น B
- `ln` = logarithm ฐานธรรมชาติ ใช้แปลงอัตราส่วนเป็นผลต่าง
- `S` = ส่วนต่างของราคาหลังแปลง log (ไม่ใช่ส่วนต่าง $ โดยตรง)

**ตัวเลขสมมติ:** ถ้า A=$120 และ B=$100 ⇒ `ln(120)−ln(100)≈0.1823`

In [ ]:
A,B=120,100
print(f'ตัวอย่างสมมติ: ln({A}) − ln({B}) = {np.log(A):.5f} − {np.log(B):.5f} = {np.log(A)-np.log(B):.5f}')
spread_ab = np.log(prices['AMAT'])-np.log(prices['LRCX'])
print('ข้อมูลจริง: 5 วันแรกของ AMAT/LRCX log spread')
display(pd.DataFrame({'ราคา AMAT':prices.AMAT,'ราคา LRCX':prices.LRCX,'log AMAT':np.log(prices.AMAT),'log LRCX':np.log(prices.LRCX),'spread':spread_ab}).head(5).round(4))

## STEP 3 — จาก 2 หุ้นสู่ 6 คู่

มีหุ้น 4 ตัว เลือกเป็นคู่ได้ `4×(4−1)÷2 = 6` คู่ ไม่ใช่ 4 คู่

**เหตุผล:** กลยุทธ์ต้องการฟังหลายความสัมพันธ์ก่อนตัดสินว่าหุ้นหนึ่งตัวดูแพง/ถูกเชิงสัมพันธ์

In [ ]:
from itertools import combinations
pairs=list(combinations(UNIVERSE,2))
print('6 Pairs:',', '.join(f'{a}/{b}' for a,b in pairs))
display(pd.DataFrame({'Pair':[f'{a}/{b}' for a,b in pairs],'Economic theme':['Semiconductor Equipment']*len(pairs)}))

## STEP 4 — Z-score บอกว่า Spread ต่างจากอดีตกี่ SD

**คำอ่านสูตร:** `Z = (Spread วันนี้ − ค่าเฉลี่ย Spread ของ 60 วันล่าสุด) ÷ ส่วนเบี่ยงเบนมาตรฐานของ Spread 60 วัน`

\[Z_{AB,t}=\frac{S_{AB,t}-\mu_{AB,t}}{\sigma_{AB,t}}\]

- `S` = spread วันนี้
- `μ` = ค่าเฉลี่ยของ spread ในหน้าต่างข้อมูลย้อนหลัง
- `σ` = Standard Deviation ของ spread ในหน้าต่างเดียวกัน
- `Z > 0` = หุ้นตัวแรก Rich เมื่อเทียบกับหุ้นตัวที่สองภายใต้โมเดล

**ตัวเลขสมมติ:** Spread=0.30, เฉลี่ย=0.10, SD=0.10 ⇒ Z=(0.30−0.10)/0.10=+2.00

In [ ]:
LOOKBACK = 60
print('ตัวอย่างสมมติ: (0.30 - 0.10) / 0.10 =', (0.30-0.10)/0.10)
rolling_mean=spread_ab.rolling(LOOKBACK).mean()
rolling_sd=spread_ab.rolling(LOOKBACK).std(ddof=1)
z_ab=(spread_ab-rolling_mean)/rolling_sd
sample=pd.DataFrame({'Spread':spread_ab,'เฉลี่ยย้อนหลัง':rolling_mean,'SD ย้อนหลัง':rolling_sd,'Z':z_ab}).dropna()
display(sample.head(5).round(4))

## STEP 5 — Flattening: หนึ่ง Pair สร้าง 2 Votes

ถ้า Z ของ AMAT/LRCX = **+2.4** แปลว่า `AMAT +2.4` (ดูแพง) และ `LRCX −2.4` (ดูถูก)

\[vote_A = +Z_{AB},\quad vote_B=-Z_{AB}\]

- `vote` = เสียงประเมิน Relative Mispricing ไม่ใช่คำสั่งส่ง Order
- เครื่องหมายบวก = Rich Candidate, เครื่องหมายลบ = Cheap Candidate

**ตัวเลขสมมติ:** +2.4 และ −2.4 รวมกันเป็น 0 เสมอสำหรับหนึ่ง Pair

In [ ]:
example_z=2.4
print(f'AMAT vote: +{example_z} | LRCX vote: {-example_z}')
zs, agg, depth, consistency = compute_signals(prices,lookback=LOOKBACK)
print('Z-score รายคู่ (ข้อมูลจริง)')
display(zs.dropna().head(5).round(3))

## STEP 6 — รวมหลาย Votes เป็นสัญญาณของหุ้นหนึ่งตัว

**สูตรเข้าใจง่าย:** `Aggregate Signal = ผลรวม votes ของหุ้น ÷ จำนวน votes ที่มีข้อมูล`

\[Signal_i = \frac{vote_{i,1}+vote_{i,2}+\cdots+vote_{i,N}}{N}\]

**ตัวอย่างสมมติ:** หุ้น A ได้ votes +2.4, +2.0, +2.1 ⇒ `(2.4+2.0+2.1)/3 = +2.1667`

**ข้อแตกต่างจากบทความ:** Notebook ใช้ *น้ำหนักเท่ากัน* แทน Sharpe-weighted votes เพื่อให้เข้าใจสูตรและไม่ใช้ผลฝึกย้อนหลังมาคัดคู่จากตัวอย่างนี้

In [ ]:
example_votes=[2.4,2.0,2.1]
print(f'ตัวอย่าง: ({" + ".join(map(str,example_votes))}) / 3 = {np.mean(example_votes):.4f}')
print('Aggregate Signal ของหุ้นทุกตัว (ข้อมูลจริง)')
display(agg.dropna().head(5).round(3))

## STEP 7 — Depth และ Consistency ทำไมสำคัญ?

**Depth = จำนวน Pair ที่ให้เสียงกับหุ้นนั้น**

**Consistency = ค่าสัมบูรณ์ของค่าเฉลี่ยเครื่องหมาย votes**

\[Consistency = \left|\frac{sign(v_1)+\cdots+sign(v_N)}{N}\right|\]

- `sign(v)` = +1 ถ้า vote บวก / −1 ถ้า vote ลบ
- `|...|` = ค่าสัมบูรณ์ (ทำให้ไม่ติดลบ)
- `N` = จำนวนคู่ที่มี votes

**ตัวเลขสมมติ:** +,+,+ ⇒ Consistency=1.00; +,+,− ⇒ |(1+1−1)/3|=0.33

**ข้อจำกัด:** 4 หุ้น ให้ Depth สูงสุด 3 จึงใช้ `min_depth=2` สำหรับห้องเรียน; งานวิจัยเต็มใช้ ≥4

In [ ]:
for vv in [[1,1,1],[1,1,-1],[1,-1]]:
 print('vote signs',vv,'→ depth',len(vv),'consistency',abs(np.mean(vv)))
print('ตัวอย่างตาราง Depth และ Consistency จากราคาจริง')
display(pd.concat({'Depth':depth,'Consistency':consistency},axis=1).dropna().head(5).round(3))

## STEP 8 — กฎ Portfolio: Long Cheap / Short Rich

- เลือกหุ้นคะแนนติดลบมากที่สุดที่ผ่าน Depth/Consistency เป็น **Long**
- เลือกหุ้นคะแนนบวกมากที่สุดที่ผ่าน Depth/Consistency เป็น **Short**
- แต่ละขามูลค่า **50% ของทุน** เพื่อมี Gross Exposure 100%
- ถ้าไม่พบทั้งสองด้าน = **ถือเงินสด**

ตัวอย่างสมมติ: Long KLAC $5,000 + Short AMAT $5,000 บนทุน $10,000

**หมายเหตุ:** ใช้ Consistency >0.5 (เคร่งครัด) และ Depth≥2 เท่านั้นสำหรับ prototype นี้

In [ ]:
MIN_DEPTH=2
MIN_CONSISTENCY=0.5
COST_BPS=5
bt,z_all,aggregates,depths,consistencies,positions,signals=build_strategy(prices,window=LOOKBACK,min_depth=MIN_DEPTH,min_consistency=MIN_CONSISTENCY,cost_bps=COST_BPS)
print('ตัวอย่าง Target Weights ของหุ้นแต่ละตัว (ก่อนเลื่อนไปใช้วันถัดไป)')
display(positions.loc[positions.abs().sum(axis=1)>0].head(6).round(2))
print('ตัวอย่าง Trade Selection (signal date)')
display(signals.head(6))

## STEP 9 — แยก Signal Date กับวันที่รับผลตอบแทน

**กฎสำคัญ:** เราเพิ่งรู้ราคาปิดของวัน t หลังจบวันแล้ว จึงห้ามใช้ Signal วันนี้เพื่ออ้างว่าได้กำไรจากราคาที่เกิดขึ้นก่อนหน้านั้น

`Position ที่ใช้คำนวณ Return วันนี้ = Target Position จากเมื่อวาน`

\[Position_{used,t}=TargetPosition_{t-1}\]

- `t` = วันปัจจุบัน
- `t−1` = วันก่อนหน้า

สคริปต์ใช้ next-bar close-to-close proxy ยังไม่ใช่การรับประกัน Fill ในตลาดจริง

In [ ]:
# สาธิตว่าต้อง shift 1 วันก่อนคูณผลตอบแทน
executed=positions.shift(1).fillna(0)
example=pd.DataFrame({'Target AMAT':positions['AMAT'],'Executed AMAT':executed['AMAT']})
display(example.loc[example['Target AMAT'].ne(0)|example['Executed AMAT'].ne(0)].head(8))

## STEP 10 — Gross Return, Turnover และ Net Return

**สูตรจำง่าย**

`Gross Return = ผลรวม(น้ำหนักหุ้นแต่ละตัว × ผลตอบแทนหุ้นตัวนั้น)`

`Turnover = ผลรวมของค่าสัมบูรณ์(น้ำหนักใหม่ − น้ำหนักเดิม)`

`Net Return = Gross Return − Turnover × 5/10,000`

ตัวอย่างสมมติ: Long 50% ได้ +2%, Short 50% ราคาหุ้นเพิ่ม +1% ⇒ Gross = 0.5×2% − 0.5×1%= +0.5%; เปิดจากพอร์ตว่าง Turnover=1 ⇒ ต้นทุน 0.05% ⇒ Net=+0.45%

**ยังไม่รวม:** Short Borrow, Financing, Dividends, Slippage และผลจริงจาก Execution

In [ ]:
gross_demo=.5*.02-.5*.01
turn_demo=1.0; cost=turn_demo*5/10000
print(f'Gross = {gross_demo:.2%}; Turnover={turn_demo}; Cost={cost:.2%}; Net = {gross_demo-cost:.2%}')
display(bt[['gross_return','turnover','net_return','equity','drawdown']].head(10).round(5))

## STEP 11 — มูลค่าพอร์ต (Equity) และ Drawdown

`Equity วันนี้ = Equity เมื่อวาน × (1 + Net Return วันนี้)`

`Drawdown = Equity วันนี้ ÷ Equity สูงสุดที่เคยเกิดขึ้น − 1`

ตัวอย่างสมมติ: ทุน 1.00, กำไรวันแรก 1% ⇒ Equity=1.01; วันถัดมาขาดทุน 2% ⇒ Equity=1.01×0.98=0.9898

ถ้าจุดสูงสุดเดิมคือ 1.01 ⇒ Drawdown=(0.9898/1.01)−1=−2%

In [ ]:
print('สมมติ: 1.01 * 0.98 =',round(1.01*.98,4))
print('Drawdown =',f'{(1.01*.98/1.01-1):.2%}')
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(bt.index,bt.equity,label='Strategy 2 net');ax.plot(bt.index,bt.benchmark_equity,label='Equal-weight buy/hold')
ax.set_title('Historical portfolio equity from local CSV');ax.legend();ax.grid(alpha=.2)
plt.show()

## STEP 12 — Sharpe Ratio และผลการทดลอง

`Sharpe (rf=0) = (ค่าเฉลี่ย Net Return รายวัน ÷ SD ของ Net Return รายวัน) × √252`

- `rf=0` = ตั้ง Risk-free Rate เท่ากับศูนย์ เพื่อการสอน
- `252` = สมมติวันซื้อขายต่อปี
- `SD` = Standard Deviation ของผลตอบแทนรายวัน

**สำคัญ:** ต้องเปรียบเทียบ Strategy 1 และ Benchmark ในช่วงเวลาเดียวกัน ไม่ใช้ตัวเลขจากคนละช่วงมาเทียบตรง ๆ

In [ ]:
print('Strategy 2 metrics (ผลขึ้นกับไฟล์ uSMART บน Mac):')
display(pd.DataFrame({'Strategy 2':metrics(bt.net_return),'Benchmark':metrics(bt.benchmark_return)}).round(4))
print('ดูวันเวลาที่ใช้:',bt.index.min(),bt.index.max())

## STEP 13 — Daily-by-Week Heatmap: ผลจริง ไม่ใช่รูปตัวอย่าง

แต่ละช่องคือ Net Return รายวัน แบ่ง Mon–Fri ในแต่ละ ISO Week ช่วยให้เห็นว่ากลยุทธ์มีความต่อเนื่องหรืออาศัยบางวันเท่านั้น

In [ ]:
daily=bt.net_return.copy();tab=pd.DataFrame({'year':daily.index.isocalendar().year.values,'week':daily.index.isocalendar().week.values,'weekday':daily.index.weekday,'return':daily.values})
pivot=tab.pivot_table(index=['year','week'],columns='weekday',values='return')*100
print('5 สัปดาห์ล่าสุด (%)')
display(pivot.tail(5).round(2))

## STEP 14 — สร้าง Interactive HTML Tear Sheet สำหรับใช้สอน

รันแล้วจะได้ Equity Curve, Drawdown, Monthly/Weekly Heatmaps, Aggregate Signals, Trade Decisions และตาราง Strategy 1/2/Benchmark

**Strategy 3 จะไม่ถูกสร้างเป็นตัวเลขสมมติ** เพราะเรายังไม่ได้พัฒนา Asset Embeddings

In [ ]:
# ผลของไฟล์นี้อ่านจากราคาเดิม ไม่ Login และไม่ส่งคำสั่งซื้อขาย
_,_,_,_,_ = run(DATA,OUT,window=LOOKBACK,depth=MIN_DEPTH,consistency=MIN_CONSISTENCY,cost_bps=COST_BPS,open_browser=False)
print('เปิดรายงานได้ที่:',OUT/'interactive_tearsheet.html')

## STEP 15 — สิ่งที่ยังพิสูจน์ไม่ได้

1. Dataset 4 หุ้นนี้ยังไม่ใช้ 13F Asset Embeddings / Point-in-Time Pair Selection
2. Pair Votes **Equal-weight** ไม่ใช่ Sharpe-weighted แบบงานวิจัยเต็ม
3. ยังไม่มี Short Borrow/Funding/Dividend/Execution Fill
4. 5 bps เป็นเพียง *สมมติฐานต้นทุน* ไม่ใช่ค่าธรรมเนียมที่ยืนยันกับ uSMART
5. Backtest ไม่ใช่หลักฐานว่ามี Edge Out-of-Sample จนกว่าจะทดสอบช่วงที่ไม่ได้ใช้พัฒนาโมเดล

**Key message:** One pair gives a comparison; a network gives context. Context is not guaranteed profit.